# Parte 6 (opcional) – ¿Le gana una regresión logística al BG/NBD?

**Autor:** Herman Andrés Echeverría Rojas · Grupo 1

**Declaración de uso de IA:** se utilizó Claude (Anthropic) como herramienta de apoyo para comprender lo que pedía cada inciso del enunciado y como asistencia al escribir y depurar el código. El análisis, la verificación de resultados y su interpretación fueron revisados por el autor.

**Referencias principales**

- Fader, P. S., Hardie, B. G. S. y Lee, K. L. (2005). "Counting Your Customers" the Easy Way: An Alternative to the Pareto/NBD Model. *Marketing Science*, 24(2), 275–284.
- Documentación de scikit-learn: `LogisticRegression`, `StratifiedKFold`, `Pipeline` y `brier_score_loss`.

En la Parte 4, Gregorio encontró que $P(Z = 1)$ del BG/NBD no ordena a los clientes mucho mejor que la recencia sola. Si lo que queremos es $P(Z = 1)$, una regresión logística con $x$, $t_x$ y $T$ es la respuesta obvia de cualquier científico de datos. Aquí la ponemos a competir (6.1) y después revisamos si la competencia fue justa (6.2–6.4). Las funciones del BG/NBD son las de `src/bgnbd.py` (`p_compra`, de Gregorio) y los parámetros se leen de `resultados/params_cdnow.json`.

In [1]:
import sys, json
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from datos import cargar_cdnow, resumen_clientes
from bgnbd import p_compra

trans = cargar_cdnow("../data/CDNOW_sample.txt")
params = json.load(open("../resultados/params_cdnow.json"))
VARIABLES = ["x", "tx", "T"]

# Calibración hasta el 30 sep 1997 y holdout de 39 semanas, como en las Partes 3 y 4
res = resumen_clientes(trans, "1997-09-30", "1998-06-30")
z = (res["x_star"] >= 1).astype(int).values
res["p_compra"] = p_compra(39, res["x"], res["tx"], res["T"], params)

print(f"{len(res)} clientes; tasa base de Z = {z.mean():.3f}")
print("Parámetros del BG/NBD:", {k: round(v, 4) for k, v in params.items()})

2357 clientes; tasa base de Z = 0.290
Parámetros del BG/NBD: {'r': 0.2426, 'alpha': 4.4136, 'a': 0.7929, 'b': 2.4259}


In [2]:
def nueva_logistica():
    # Sin penalización, como pide el enunciado. El escalador va dentro del pipeline para que,
    # en la validación cruzada, la media y la DE se calculen solo con los pliegues de entrenamiento.
    return make_pipeline(StandardScaler(), LogisticRegression(C=np.inf, max_iter=1000))


def metricas(y, prob):
    return {"AUC": roc_auc_score(y, prob), "Brier": brier_score_loss(y, prob),
            "prob. promedio": np.mean(prob), "tasa real": np.mean(y)}


def tabla_deciles(prob, y):
    # rank(method="first") rompe empates para que cada decil tenga el mismo número de clientes
    decil = pd.qcut(pd.Series(prob).rank(method="first"), 10, labels=range(1, 11))
    g = pd.DataFrame({"prob": np.asarray(prob), "y": np.asarray(y)}).groupby(decil.values, observed=True)
    return pd.DataFrame({"prob. promedio": g["prob"].mean(), "fracción que compró": g["y"].mean()}).rename_axis("decil")


def grafica_calibracion(tablas, titulo, archivo):
    colores = {"Logística": "#4C72B0", "BG/NBD": "#C44E52"}
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.plot([0, 1], [0, 1], color="gray", ls=":", label="Calibración perfecta")
    for nombre, t in tablas.items():
        ax.plot(t["prob. promedio"], t["fracción que compró"], marker="o",
                color=colores.get(nombre.split(" ")[0], None), label=nombre)
    ax.set(title=titulo, xlabel="Probabilidad promedio del decil", ylabel="Fracción que compró",
           xlim=(0, 1), ylim=(0, 1))
    ax.legend(loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.savefig(f"../figs/{archivo}", dpi=150)
    plt.show()

## 6.1 La comparación ingenua

Se ajusta una regresión logística de $Z_i$ contra $x$, $t_x$ y $T$ estandarizados y sin penalización (`LogisticRegression(C=np.inf)`). Se evalúa con validación cruzada estratificada de 5 pliegues **sobre clientes**, repetida con 20 semillas (las 20 primeras de la lista de la propuesta), es decir, 100 particiones.

En cada pliegue de prueba se calculan el AUC y el Brier score de la logística y de $P(Z = 1)$ del BG/NBD (inciso 4.3) **sobre los mismos clientes**. El BG/NBD usa los mismos parámetros en todos los pliegues: se estimó solo con $(x, t_x, T)$ de calibración y nunca vio una etiqueta $Z$, así que no hay nada que reajustar.

In [3]:
SEMILLAS = [77785, 14045, 114526, 85967, 85603, 2452, 175326, 32532, 155374, 101914,
            37119, 190629, 22054, 21663, 77709, 21226, 49984, 119397, 169546, 6120]
X = res[VARIABLES].values
pz = res["p_compra"].values

filas, oof = [], np.zeros((len(SEMILLAS), len(z)))
for i, semilla in enumerate(SEMILLAS):
    for entrena, prueba in StratifiedKFold(5, shuffle=True, random_state=semilla).split(X, z):
        prob = nueva_logistica().fit(X[entrena], z[entrena]).predict_proba(X[prueba])[:, 1]
        oof[i, prueba] = prob
        filas.append({"semilla": semilla,
                      "AUC logística": roc_auc_score(z[prueba], prob),
                      "AUC BG/NBD": roc_auc_score(z[prueba], pz[prueba]),
                      "Brier logística": brier_score_loss(z[prueba], prob),
                      "Brier BG/NBD": brier_score_loss(z[prueba], pz[prueba])})
cv = pd.DataFrame(filas)

resumen_cv = pd.DataFrame({"media": cv.drop(columns="semilla").mean(), "DE": cv.drop(columns="semilla").std()})
print(f"La logística tiene mayor AUC en {(cv['AUC logística'] > cv['AUC BG/NBD']).mean():.0%} de los 100 pliegues "
      f"y menor Brier en {(cv['Brier logística'] < cv['Brier BG/NBD']).mean():.0%}.")
resumen_cv.round(4)

La logística tiene mayor AUC en 93% de los 100 pliegues y menor Brier en 94%.


,media,DE
AUC logística,0.7791,0.0217
AUC BG/NBD,0.7570,0.0224
Brier logística,0.1562,0.0077
Brier BG/NBD,0.1612,0.0064


> **¿Quién gana la comparación ingenua?**

La logística, y no por suerte. Su AUC promedio es ≈ 0.779 frente a ≈ 0.757 del BG/NBD, y su Brier score es ≈ 0.156 frente a ≈ 0.161. Gana en AUC en el 93 % de los 100 pliegues y en Brier en el 94 %, así que la diferencia no se explica por la partición. La DE entre pliegues (≈ 0.02 en AUC) es del mismo orden que la diferencia, pero como en cada pliegue los dos modelos se evalúan sobre los mismos clientes, lo que importa es que la logística sale arriba casi siempre.